# 10 — POS Tagging
**Goal:** Label every word with its part of speech.

Part-of-speech tagging assigns each token a grammatical class — noun, verb, adjective, determiner. It is the first stage in the pipeline that reasons about *syntax* rather than strings: the same word can be a noun in one sentence and a verb in another, and only context decides. Modern taggers (spaCy's) are sequence models trained on annotated text, so they are fast, accurate, and need no rule lists.

**Why it matters for resumes / ATS:** resume bullets are written as action-verb leads — "Developed…", "Reduced…", "Collaborated…". POS is what lets software find the action word in each bullet, verify that a bullet starts with a verb, and separate skills (nouns) from actions. It is also the input Ch. 08's lemmatizer needs and the foundation Ch. 11's dependency parsing builds on.

| POS Tag | Full Name | Example | Resume Context |
|---|---|---|---|
| `NOUN` | Noun | `engineer`, `pipeline` | Skills, roles |
| `VERB` | Verb | `developed`, `reduced` | Action bullets |
| `ADJ` | Adjective | `senior`, `expert` | Qualifiers |
| `PROPN` | Proper noun | `Python`, `Google` | Brands, names |
| `AUX` | Auxiliary | `is`, `was`, `will` | Tense markers |
| `ADP` | Adposition | `with`, `at`, `in` | Context signals |

## 1. Context Determines POS

A word's POS is not a property of the word — it is a property of the sentence. `book` is a verb in "I will book a flight" and a noun in "That book is interesting". A tagger resolves this from surrounding context; a rule or regex cannot.

| Sentence | `book` POS | Why |
|---|---|---|
| "I will **book** a flight" | VERB | After modal `will`, expects verb |
| "Please **book** the restaurant" | VERB | Imperative, command form |
| "That **book** is interesting" | NOUN | After determiner `That`, expects noun |

In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm")

print(f"{'Sentence':<40} {'Token':<10} {'POS'}")
print("-" * 60)
for s in ["I will book a flight", "Please book the restaurant", "That book is interesting"]:
    doc = nlp(s)
    for t in doc:
        if t.text == "book":
            print(f"{s:<40} {t.text:<10} {t.pos_}")

**Observation:** The same token `book` is tagged VERB, VERB, and NOUN across the three sentences — identical surface form, different grammatical role. This context-sensitivity is why POS tagging sits before lemmatization and dependency parsing in the pipeline.

## 2. Action Verbs from Resume Bullets

Resume style guides demand bullets that open with a past-tense action verb. POS tagging turns that stylistic rule into an automatable check: parse the bullet, look at the first token's POS, and flag bullets that fail.

| Bullet | First Verb | Starts with Verb? | Status |
|---|---|---|---|
| "Developed ML models..." | `Developed` | ✓ | Pass |
| "Led a team of 5..." | `Led` | ✓ | Pass |
| "Reduced inference latency..." | `None` | ✗ | Fail (see note) |
| "Collaborated with cross-functional..." | `Collaborated` | ✓ | Pass |

**⚠️ Note:** `Reduced` is tagged as ADJ (adjective) here because spaCy reads "reduced latency" as a modified noun phrase. The heuristic misses a perfectly good action bullet — treat POS-based checks as signals for review, not oracles.

In [ ]:
bullets = [
    "Developed ML models using TensorFlow",
    "Led a team of 5 data scientists",
    "Reduced inference latency by 40%",
    "Collaborated with cross-functional teams",
]

print(f"{'Status':<8} {'Bullet':<45} {'First Verb':<15} {'Starts Verb?'}")
print("-" * 80)
for b in bullets:
    doc = nlp(b)
    first_verb = next((t.text for t in doc if t.pos_ == "VERB"), None)
    starts_verb = doc[0].pos_ == "VERB"
    print(f"{'✓' if starts_verb else '✗':<8} {b:<45} {str(first_verb):<15} {starts_verb}")

**Try it:** the `Reduced` row is the interesting failure: spaCy tags the past participle as an adjective here, so the heuristic misses it. The check is cheap and useful, but it inherits the tagger's errors.

## Key Insight

**POS turns words into grammar, and grammar into structure.**

Tagging is the pivot of the whole pipeline: it gives the lemmatizer its POS signal (Ch. 08), tells dependency parsing which words to connect (Ch. 11), and enables cheap, interpretable quality checks like "every bullet starts with a verb" — checks an ATS can run over thousands of resumes without any ML beyond the tagger itself.

The catch this chapter showed: taggers are statistical, and their errors propagate. A mis-tagged participle silently flips a bullet's quality flag. Next, Ch. 11 — Dependency Parsing — builds on these labels to recover who-did-what, the structure that turns tagged words into extractable facts.